# Análise de latência e erros

In [17]:
%load_ext autoreload
%autoreload 2

In [18]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent  # if notebook is in notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import time
import tracemalloc
import numpy as np
import pandas as pd

from src.features import build_ranking_features
from src.paths import ARTIFACTS_DIR, PROC_DIR
from src.ranking import load_ranker, score_candidates
from src.retrieval import BM25Retriever
from src.evaluation import map_relevance, evaluate_ranking, evaluate_retrieval

In [19]:
products = pd.read_parquet(PROC_DIR / "products.parquet")
queries = pd.read_parquet(PROC_DIR / "queries.parquet")
labels = pd.read_parquet(PROC_DIR / "labels.parquet").copy()
splits = pd.read_parquet(PROC_DIR / "splits.parquet")

labels["relevance"] = map_relevance(labels["label"])

test_qids = set(splits.loc[splits["split"] == "test", "query_id"])
test_queries = queries[queries["query_id"].isin(test_qids)].copy()
test_labels = labels[labels["query_id"].isin(test_qids)].copy()

ranker, feat_cols = load_ranker(ARTIFACTS_DIR / "hgb_ranker.joblib")

print(f"Loaded {len(test_queries)} test queries and {len(test_labels)} test label rows.")

Loaded 72 test queries and 32923 test label rows.


## Latência

- Index build time + peak Python memory

In [25]:
tracemalloc.start()
t0 = time.perf_counter()
retriever = BM25Retriever().fit(products)
index_build_sec = time.perf_counter() - t0
_, peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()

In [26]:
print(f"Index build time: {index_build_sec:.3f}s | Peak traced memory: {peak_bytes / (1024**2):.1f} MB")

Index build time: 66.539s | Peak traced memory: 816.7 MB


- Per-query latency across evaluation queries

In [27]:
sample_queries = test_queries["query"].astype(str).tolist()

# Warm-up call
_w_df = pd.DataFrame([{"query_id": 0, "query": sample_queries[0]}])
_w_cands = retriever.retrieve(_w_df, top_k=100)
_w_feat = build_ranking_features(candidates=_w_cands, queries=_w_df, products=products, labels=None)
_ = score_candidates(ranker, _w_feat, feat_cols)

ret_ms, rank_ms, total_ms = [], [], []
for idx, q_text in enumerate(sample_queries):
    q_df = pd.DataFrame([{"query_id": idx, "query": q_text}])

    t_ret_start = time.perf_counter()
    cands = retriever.retrieve(q_df, top_k=100)
    t_ret_end = time.perf_counter()

    t_rank_start = time.perf_counter()
    feat = build_ranking_features(candidates=cands, queries=q_df, products=products, labels=None)
    _ = score_candidates(ranker, feat, feat_cols)
    t_rank_end = time.perf_counter()

    r_dur = (t_ret_end - t_ret_start) * 1000.0
    k_dur = (t_rank_end - t_rank_start) * 1000.0
    ret_ms.append(r_dur)
    rank_ms.append(k_dur)
    total_ms.append(r_dur + k_dur)

def summarize_ms(values: list[float]) -> dict[str, float]:
    arr = np.asarray(values, dtype=np.float64)
    return {
        "p50_ms": float(np.percentile(arr, 50)),
        "p95_ms": float(np.percentile(arr, 95)),
        "mean_ms": float(np.mean(arr)),
    }

summary = pd.DataFrame(
    [
        {"stage": "1. BM25 Retrieval", **summarize_ms(ret_ms)},
        {"stage": "2. Features + HGB Scoring", **summarize_ms(rank_ms)},
        {"stage": "Total End-to-End (per query)", **summarize_ms(total_ms)},
    ]
)
print(summary.to_string(index=False))

                       stage   p50_ms    p95_ms    mean_ms
           1. BM25 Retrieval 146.4742 879.98901 316.151574
   2. Features + HGB Scoring  89.9084 151.83311 103.235494
Total End-to-End (per query) 266.2547 960.39652 419.387068


## Análise de erros

In [28]:
retriever = BM25Retriever().fit(products)
test_cands = retriever.retrieve(test_queries, top_k=100)
test_feat = build_ranking_features(
    candidates=test_cands,
    queries=test_queries,
    products=products,
    labels=test_labels,
)
test_scored = score_candidates(ranker, test_feat, feat_cols)

In [30]:
q_records = []
for qid, q_group in test_scored.groupby("query_id", sort=False):
    q_labels = test_labels[test_labels["query_id"] == qid]
    q_cands = test_cands[test_cands["query_id"] == qid]

    rec_metrics = evaluate_retrieval(q_cands, q_labels, ks=[100])
    hgb_metrics = evaluate_ranking(q_group, ks=[10])

    bm25_view = q_group[["query_id", "bm25_score", "relevance"]].rename(
        columns={"bm25_score": "score"}
    )
    bm25_metrics = evaluate_ranking(bm25_view, ks=[10])

    q_text = (
        q_group["query"].iloc[0]
        if "query" in q_group.columns
        else test_queries.loc[test_queries["query_id"] == qid, "query"].iloc[0]
    )

    q_records.append(
        {
            "query_id": qid,
            "query": q_text,
            "n_relevant_total": int((q_labels["relevance"] > 0).sum()),
            "n_relevant_in_topk": int(
                q_group.loc[q_group["relevance"] > 0, "product_id"].nunique()
            ),
            "recall@100": rec_metrics["recall@100"],
            "bm25_ndcg@10": bm25_metrics["ndcg@10"],
            "hgb_ndcg@10": hgb_metrics["ndcg@10"],
            "ndcg_delta": hgb_metrics["ndcg@10"] - bm25_metrics["ndcg@10"],
        }
    )

per_query_df = pd.DataFrame(q_records)

In [31]:
print("Zero-recall@100 queries:", int((per_query_df["recall@100"] == 0).sum()), "/", len(per_query_df))
print("Queries where HGB beat BM25:", int((per_query_df["ndcg_delta"] > 0).sum()), "/", len(per_query_df))
print("Queries where HGB trailed BM25:", int((per_query_df["ndcg_delta"] < 0).sum()), "/", len(per_query_df))

worst_df = per_query_df.sort_values(["hgb_ndcg@10", "recall@100"]).head(10)
print("\nWorst 10 queries by HGB NDCG@10:")
print(worst_df.to_string(index=False))

Zero-recall@100 queries: 3 / 72
Queries where HGB beat BM25: 38 / 72
Queries where HGB trailed BM25: 14 / 72

Worst 10 queries by HGB NDCG@10:
 query_id                                      query  n_relevant_total  n_relevant_in_topk  recall@100  bm25_ndcg@10  hgb_ndcg@10  ndcg_delta
      174                              star wars rug                 7                   0    0.000000      0.000000     0.000000    0.000000
      222                    small loving roomtables                36                   0    0.000000      0.000000     0.000000    0.000000
      383                               minnestrista                 3                   0    0.000000      0.000000     0.000000    0.000000
      173                        bedroom accessories               204                   1    0.004902      0.430677     0.000000   -0.430677
      400        small ladies rocker swivel recliner                24                  12    0.500000      0.000000     0.136985    0.136985
     

In [16]:
target_qids = [173, 76, 485]
name_map = (
    products.drop_duplicates("product_id")
    .set_index("product_id")["product_name"]
    .to_dict()
)

for qid in target_qids:
    q_text = test_queries.loc[test_queries["query_id"] == qid, "query"].iloc[0]
    sub = (
        test_scored[test_scored["query_id"] == qid]
        .sort_values(["score", "bm25_score"], ascending=[False, False])
        .head(5)
    )
    print(f"\n=== Query {qid}: '{q_text}' ===")
    for rank_i, r in enumerate(sub.to_dict("records"), start=1):
        pname = str(name_map.get(r["product_id"], r["product_id"]))[:65]
        print(
            f"  {rank_i}. [{r['product_id']}] {pname} | "
            f"score={r['score']:.4f} | bm25={r['bm25_score']:.3f} | rel={r['relevance']}"
        )


=== Query 173: 'bedroom accessories' ===
  1. [10851] trundle unit | score=1.4961 | bm25=6.077 | rel=0
  2. [37855] 7 drawers dresser , vertical storage tower with shelves , fabric  | score=1.4638 | bm25=5.385 | rel=0
  3. [37853] 9-drawer extra wide dresser storage tower - sturdy steel frame ,  | score=1.4620 | bm25=5.453 | rel=0
  4. [37854] vertical dresser storage tower - sturdy steel frame , wood top ,  | score=1.4368 | bm25=5.481 | rel=0
  5. [10949] lift bed risers | score=1.2116 | bm25=5.943 | rel=0

=== Query 76: '46 inch closet door' ===
  1. [37877] terrace antique brass passage hall/closet door knob with universa | score=1.3920 | bm25=18.885 | rel=0
  2. [30069] sliding closet mirror door top guide | score=1.1842 | bm25=11.992 | rel=1
  3. [42273] sliding closet door finger pull | score=1.1734 | bm25=13.461 | rel=0
  4. [42271] bi-fold closet door snugger | score=1.1568 | bm25=12.447 | rel=0
  5. [30073] bi-fold closet 2-door snugger | score=1.1359 | bm25=12.030 | rel=0

=

- Custo de indexação

In [32]:
fit_times_ms = []
peak_mem_mb = []

for _ in range(5):
    tracemalloc.start()
    t0 = time.perf_counter()
    bm25 = BM25Retriever().fit(products)
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    fit_times_ms.append(elapsed_ms)
    peak_mem_mb.append(peak_bytes / (1024 ** 2))

n_products = len(products)
n_vocab = len(bm25.idf_) if hasattr(bm25, "idf_") else None
print(f"Catalog size: {n_products:,} products | Vocab size: {n_vocab}")
print(f"Indexing wall time: mean={np.mean(fit_times_ms):.1f} ms | p50={np.percentile(fit_times_ms, 50):.1f} ms")
print(f"Peak Python memory during fit: {np.mean(peak_mem_mb):.1f} MB")

Catalog size: 42,994 products | Vocab size: None
Indexing wall time: mean=85151.9 ms | p50=65041.9 ms
Peak Python memory during fit: 816.7 MB
